# M02.02 — Tool Calling

Tools let the model do what it cannot do on its own, such as fetching live data or computing with guaranteed precision. We will:

1. Ask questions an LLM cannot answer correctly without tools.
2. Define a tool (a sum function) and run a two-turn conversation.
3. Write the schema of a second tool (current weather).
4. Chain tool calls into a fixed pipeline: one tool per turn, in an order we choose.
5. Add a third tool (mean) and combine weather and math in a three-step pipeline.

After every call we print its cost.

## Setup

In [ ]:
import json
import random

from agentic_workshop.client import get_client
from agentic_workshop.config import CHAT_MODEL, REASONING_OFF
from agentic_workshop.costs import print_cost, usage_cost

client = get_client()
MODEL = CHAT_MODEL

## 1. Questions an LLM answers poorly without tools

### 1.1 Current weather in Geneva

The model has no access to live weather data, so it will either refuse or hallucinate.

In [ ]:
weather_prompt = "What is the current weather in Geneva?"

response_weather = client.responses.create(model=MODEL, input=weather_prompt)

print(response_weather.output_text)
print_cost(response_weather.usage)

### 1.2 Sum of very large numbers

LLMs are not calculators. Given three very large numbers, they often make arithmetic mistakes. Compare the LLM's answer to Python's.

In [ ]:
big_numbers = [92_641_222_159_992_181_531, 8_284_817_218_129_892_289, 1_193_875_289_374_221_828]
sum_prompt = f"What is the sum of these numbers: {big_numbers}? Output only the number."

response_sum = client.responses.create(
    model=MODEL,
    input=sum_prompt,
    reasoning=REASONING_OFF,
)

print(f"LLM answer:  {response_sum.output_text}")
print(f"\nCorrect sum: {sum(big_numbers)}\n\n")
print_cost(response_sum.usage)

## 2. Adding tools to the LLM

A tool is made of two parts:
1. A **schema** that tells the model the tool exists, what it does, and which arguments it expects.
2. A **Python function** that runs when the model asks to call the tool.

We used the first part in M02.01 to extract data, but no function ran: we only read the arguments. Here we run the function and send its result back to the model, which then answers. The conversation becomes multi-turn: the model asks for one or more tool calls, we return the results, and the model produces a final answer.

### 2.1 Implementing the `sum` tool

We implement a `sum_numbers` tool that accepts an arbitrary list of numbers.

In [ ]:
def sum_numbers(numbers: list[float]) -> float:
    """Return the sum of an arbitrary list of numbers."""
    return sum(numbers)


sum_tool_schema = {
    "type": "function",
    "name": "sum_numbers",
    "description": "Compute the sum of a list of numbers.",
    "parameters": {
        "type": "object",
        "properties": {
            "numbers": {
                "type": "array",
                "items": {"type": "number"},
                "description": "A list of numbers to add together."
            }
        },
        "required": ["numbers"],
        "additionalProperties": False
    },
    "strict": True,
}

### 2.2 Two-turn conversation for the sum

A tool-calling conversation needs at least two turns:
- **Turn 1**: the user asks the question; the model replies with a `function_call` item.
- **Turn 2**: we run the tool and send the result back as a `function_call_output` item, with the same `call_id`; the model answers the user.

We use the same three large numbers as before.

In [ ]:
messages = [{"role": "user", "content": f"What is the sum of these numbers: {big_numbers}?"}]

# Turn 1: the model decides to call sum_numbers
response_turn_1 = client.responses.create(model=MODEL, input=messages, tools=[sum_tool_schema], reasoning=REASONING_OFF)

# The output items (reasoning, function calls) go back into the conversation as they are
messages += response_turn_1.output

print("Turn 1 tool calls:")
for item in response_turn_1.output:
    if item.type == "function_call":
        print(f"  {item.name}({item.arguments})")
print_cost(response_turn_1.usage)

In [ ]:
# Execute every tool call the model requested and append the results.
for item in response_turn_1.output:
    if item.type != "function_call":
        continue
    if item.name == "sum_numbers":
        args = json.loads(item.arguments)
        output = str(sum_numbers(args["numbers"]))
    else:
        output = "Error: unknown function"
    messages.append({"type": "function_call_output", "call_id": item.call_id, "output": output})

# Turn 2: the model produces the final answer
response_turn_2 = client.responses.create(model=MODEL, input=messages, tools=[sum_tool_schema])
messages += response_turn_2.output

print("Turn 2 final answer:")
print(response_turn_2.output_text)
print_cost(response_turn_2.usage)

## 3. Exercise 1: the weather tool

We provide a stub function that returns weather data for a city. Your task is to **write the tool schema** so the LLM knows how to call it.

The function returns a JSON string. In a real application it would call a weather API (e.g. Open-Meteo).

Think about why we return JSON rather than an unlabelled tuple of values.

In [ ]:
def get_current_weather(city: str) -> str:
    """Return a short weather report for the requested city."""
    conditions = ["sunny", "cloudy", "rainy", "partly cloudy"]
    temp = random.randint(5, 30)
    condition = random.choice(conditions)
    return json.dumps({
        "city": city,
        "temperature_c": temp,
        "condition": condition
    })

In [ ]:
# TODO (Exercise 1): write the schema of the get_current_weather tool
# Hint: sum_tool_schema in section 2.1 has the same structure
weather_tool_schema = {
    # TODO: Fill this in!
}

In [ ]:
# SOLUTION
weather_tool_schema = {
    "type": "function",
    "name": "get_current_weather",
    "description": "Get the current weather in a city.",
    "parameters": {
        "type": "object",
        "properties": {
            "city": {"type": "string", "description": "The name of the city, e.g. Geneva."}
        },
        "required": ["city"],
        "additionalProperties": False
    },
    "strict": True,
}

Run the two cells below. The conversation has at least two turns: the model asks for the tool, we run it, and the model answers.

In [ ]:
weather_messages = [{"role": "user", "content": "What is the current weather in Geneva?"}]

weather_response_1 = client.responses.create(model=MODEL, input=weather_messages, tools=[weather_tool_schema])
weather_messages += weather_response_1.output

print("Turn 1 tool calls:")
for item in weather_response_1.output:
    if item.type == "function_call":
        print(f"  {item.name}({item.arguments})")
print_cost(weather_response_1.usage)

In [ ]:
for item in weather_response_1.output:
    if item.type != "function_call":
        continue
    if item.name == "get_current_weather":
        args = json.loads(item.arguments)
        output = get_current_weather(args["city"])
    else:
        output = "Error: unknown function"
    weather_messages.append({"type": "function_call_output", "call_id": item.call_id, "output": output})

weather_response_2 = client.responses.create(model=MODEL, input=weather_messages, tools=[weather_tool_schema], reasoning=REASONING_OFF)
weather_messages += weather_response_2.output

print("Turn 2 final answer:")
print(weather_response_2.output_text)
print_cost(weather_response_2.usage)

## 4. A fixed pipeline of tool calls

Writing out every turn by hand repeats the same code. We wrap one turn in a helper, `run_tool_step`:
1. Send the conversation to the model, offering it **only one tool**, and require a call to it (`tool_choice="required"`).
2. Run every call the model made (it can call the tool several times in one turn, e.g. once per city).
3. Append the results to the conversation.

`final_answer` then sends the conversation without any tools, so the model has to answer.

**We** decide which tool is offered at each step and how many steps there are; the model only fills in the arguments. Letting the model choose the tools and the number of turns itself makes it an agent: that is the topic of M04.

In [ ]:
def run_tool_step(messages: list, tool_schema: dict, tool_fn) -> float:
    """One turn: offer the model a single tool, run every call it makes and append the results. Returns the cost."""
    response = client.responses.create(model=MODEL, input=messages, tools=[tool_schema], tool_choice="required")
    messages += response.output

    for call in response.output:
        if call.type != "function_call":
            continue
        args = json.loads(call.arguments)
        result = tool_fn(**args)
        output = result if isinstance(result, str) else json.dumps(result)
        messages.append({"type": "function_call_output", "call_id": call.call_id, "output": output})
        print(f"Tool call: {call.name}({args}) -> {output}")

    print_cost(response.usage)
    return usage_cost(response.usage)["cost_usd"]


def final_answer(messages: list) -> float:
    """Last turn: no tools, the model answers from the conversation. Returns the cost."""
    response = client.responses.create(model=MODEL, input=messages)
    messages += response.output

    print("Final answer:")
    print(response.output_text)
    print_cost(response.usage)
    return usage_cost(response.usage)["cost_usd"]

In [ ]:
# The sum of section 2 as a two-step pipeline: one tool turn, then the answer
sum_messages = [{"role": "user", "content": f"What is the sum of these numbers: {big_numbers}?"}]

run_tool_step(sum_messages, sum_tool_schema, sum_numbers)
final_answer(sum_messages)

## 5. Exercise 2: a `mean` tool and a multi-step task

Write a tool from scratch that computes the mean of a list of numbers: `mean_numbers(numbers: list[float])`.

Then ask the model: **"Find the current weather in Geneva, Bern, and Zurich, then calculate the average temperature."**

We run it as a fixed pipeline of three turns, each offering only the tool it needs:
1. **Fetch**: offer `get_current_weather`; the model calls it once per city.
2. **Average**: offer `mean_numbers`; the model reads the three temperatures from the conversation and passes them in.
3. **Answer**: no tools; the model writes the summary.

In [ ]:
# TODO (Exercise 2): implement the mean_numbers function and its schema
def mean_numbers(numbers: list[float]) -> float:
    """Return the arithmetic mean of a list of numbers."""
    # TODO: Fill this in!
    pass


mean_tool_schema = {
    # TODO: Fill this in!
}

In [ ]:
# SOLUTION
def mean_numbers(numbers: list[float]) -> float:
    """Return the arithmetic mean of a list of numbers."""
    return sum(numbers) / len(numbers)


mean_tool_schema = {
    "type": "function",
    "name": "mean_numbers",
    "description": "Compute the arithmetic mean of a list of numbers.",
    "parameters": {
        "type": "object",
        "properties": {
            "numbers": {
                "type": "array",
                "items": {"type": "number"},
                "description": "A list of numbers to average."
            }
        },
        "required": ["numbers"],
        "additionalProperties": False
    },
    "strict": True,
}

In [ ]:
history = [{"role": "user", "content": "Find the current weather in Geneva, Bern, and Zurich, then calculate the average temperature and give an overview of the weather conditions."}]

total = 0.0
total += run_tool_step(history, weather_tool_schema, get_current_weather)  # Turn 1: fetch
total += run_tool_step(history, mean_tool_schema, mean_numbers)  # Turn 2: average
total += final_answer(history)  # Turn 3: answer

### Inspect the full multi-turn conversation

The history alternates between the user message, the model's output items (hidden reasoning, function calls, messages) and our tool results.

In [ ]:
def describe(item) -> str:
    if isinstance(item, dict):  # the items we wrote
        if item.get("role") == "user":
            return f"User: {item['content']}"
        return f"Tool result [{item['call_id']}]: {item['output']}"
    if item.type == "function_call":  # the items the model wrote
        return f"Assistant [tool call]: {item.name}({item.arguments})"
    if item.type == "message":
        return f"Assistant: {item.content[0].text}"
    return f"Assistant [{item.type}]"


for i, item in enumerate(history):
    print(f"{i}. {describe(item)}\n")

## Cost so far

`total` summed the cost of the three turns of the pipeline.

In [ ]:
print(f"Total cost of the weather pipeline: ${total:.6f}")

## Takeaways

- Tools let an LLM overcome its built-in limits: live data, exact arithmetic, external APIs.
- A tool needs both a schema (for the model) and a function (for your code).
- Tool calling is multi-turn: the model requests a tool call, our code runs it, and the result goes back as a `function_call_output`.
- Chaining tool calls in a fixed order, offering only the tool needed at each step, keeps the number of turns and the cost predictable. Letting the model pick the tools and decide when to stop makes it an agent: see M04.

## Extensions

1. Replace the weather stub with a real API call (e.g. Open-Meteo or another free weather service).
2. Add a `convert_temperature` tool and ask: "What is the average temperature in Geneva, Bern, and Zurich in Fahrenheit?"
3. Add error handling to `run_tool_step` so that a failing tool returns a useful message to the model instead of crashing.
4. `run_tool_step` runs the tool calls of one turn one after the other. Run them concurrently (e.g. with `concurrent.futures`) and measure whether latency drops.